In [10]:
"""

Grafo não direcionado com rotas aéreas e 8 vértices (aeroportos) e 11 arestas.

- BFS e DFS implementados com lista de adjacência

- Simulação de indisponibilidade de conexões

- Caminho antes/depois do bloqueio

"""

from collections import deque



# ---------- 1. MODELO ----------

# Lista de adjacência com siglas de aeroportos.

ARESTAS = [

  ('CWB', 'GRU'), ('CWB', 'GYN'), ('GRU', 'GYN'), ('GRU', 'CGB'), ('GYN', 'CGB'),

  ('CGB', 'BSB'), ('CGB', 'BHZ'), ('BSB', 'BHZ'),

  ('BHZ', 'GIG'), ('BHZ', 'SSA'), ('GIG', 'SSA'),

]





class Grafo:

  def __init__(self, arestas):

    self.adj = {}

    self.bloqueadas = set() # conexões indisponíveis (frozenset de 2 vértices)

    for a, b in arestas:

      self.adj.setdefault(a, []).append(b)

      self.adj.setdefault(b, []).append(a)



  def bloquear(self, a, b):

    self.bloqueadas.add(frozenset((a, b)))



  def desbloquear(self, a, b):

    self.bloqueadas.discard(frozenset((a, b)))



  def vizinhos(self, v):

    return [w for w in self.adj[v] if frozenset((v, w)) not in self.bloqueadas]



  # ---------- 2. BFS ----------

  def bfs(self, origem, destino):

    pai = {origem: None}

    fila = deque([origem])

    ordem = []

    while fila:

      v = fila.popleft()

      ordem.append(v)

      if v == destino:

        return self._caminho(pai, destino), ordem

      for w in self.vizinhos(v):

        if w not in pai:

          pai[w] = v

          fila.append(w)

    return None, ordem



  # ---------- 3. DFS ----------

  def dfs(self, origem, destino):

    pai = {origem: None}

    pilha = [origem]

    visitados = set()

    ordem = []

    while pilha:

      v = pilha.pop()

      if v in visitados:

        continue

      visitados.add(v)

      ordem.append(v)

      if v == destino:

        return self._caminho(pai, destino), ordem

      for w in reversed(self.vizinhos(v)):

        if w not in visitados:

          pai.setdefault(w, v)

          pilha.append(w)

    return None, ordem



  @staticmethod

  def _caminho(pai, destino):

    caminho = []

    v = destino

    while v is not None:

      caminho.append(v)

      v = pai[v]

    return caminho[::-1]



# ---------- 4. SIMULAÇÃO ----------

def txt(vertices):

  return " -> ".join(str(v) for v in vertices)



def mostrar(titulo, g, origem, destino):

  print(f"\n--- {titulo} ({origem} -> {destino}) ---")

  for nome, algoritmo in (("BFS", g.bfs), ("DFS", g.dfs)):

    caminho, ordem = algoritmo(origem, destino)

    if caminho:

      print(f"{nome}: {txt(caminho)} [{len(caminho) - 1} conexões]")

    else:

      print(f"{nome}: nenhum caminho disponível")

    print(f"  ordem de visita: {', '.join(str(v) for v in ordem)}")



if __name__ == "__main__":

  g = Grafo(ARESTAS)

  print(f"Vértices: {len(g.adj)} | Arestas: {len(ARESTAS)}")



  origem, destino = 'CWB', 'SSA'



  # Cenário 1: uma conexão indisponível, com desvio possível

  mostrar("ANTES do bloqueio", g, origem, destino)

  g.bloquear('CGB', 'BHZ')

  print("\n>>> Conexão indisponível: CGB <-> BHZ")

  mostrar("DEPOIS do bloqueio", g, origem, destino)



  # Cenário 2: duas conexões indisponíveis separam o grafo em duas partes

  g.bloquear('CGB', 'BSB')

  print("\n>>> Conexões indisponíveis: CGB <-> BHZ e CGB <-> BSB")

  mostrar("Cenário extremo (grafo desconectado)", g, origem, destino)

Vértices: 8 | Arestas: 11

--- ANTES do bloqueio (CWB -> SSA) ---
BFS: CWB -> GRU -> CGB -> BHZ -> SSA [4 conexões]
  ordem de visita: CWB, GRU, GYN, CGB, BSB, BHZ, GIG, SSA
DFS: CWB -> GRU -> CGB -> BHZ -> SSA [4 conexões]
  ordem de visita: CWB, GRU, GYN, CGB, BSB, BHZ, GIG, SSA

>>> Conexão indisponível: CGB <-> BHZ

--- DEPOIS do bloqueio (CWB -> SSA) ---
BFS: CWB -> GRU -> CGB -> BSB -> BHZ -> SSA [5 conexões]
  ordem de visita: CWB, GRU, GYN, CGB, BSB, BHZ, GIG, SSA
DFS: CWB -> GRU -> CGB -> BSB -> BHZ -> SSA [5 conexões]
  ordem de visita: CWB, GRU, GYN, CGB, BSB, BHZ, GIG, SSA

>>> Conexões indisponíveis: CGB <-> BHZ e CGB <-> BSB

--- Cenário extremo (grafo desconectado) (CWB -> SSA) ---
BFS: nenhum caminho disponível
  ordem de visita: CWB, GRU, GYN, CGB
DFS: nenhum caminho disponível
  ordem de visita: CWB, GRU, GYN, CGB
